# Tugas 4: Preprocessing Teks, Ekstraksi Fitur TF-IDF & Pengindeksan Terbalik (Inverted Index)
### Mata Kuliah: Temu Kembali Informasi (TKI) [SUBCPMK-4]

---

**Identitas Kelompok:**
- **Nama Anggota 1:** Akhmad Zamri Ardani (NIM: 202310370311406)
- **Nama Anggota 2:** M Rifqi Dzaky Arrafi (NIM: 202310370311441)
- **Program Studi:** Informatika, Fakultas Teknik, Universitas Muhammadiyah Malang
- **Dataset:** Koleksi Putusan Pidana Narkotika PN Jember 2023 (Dataset Mandiri Tugas 3)
- **Cakupan Teks:** Seluruh bagian dokumen putusan secara utuh (Full-Text)

---

### Alur Pemrosesan TKI (Information Retrieval Pipeline):
1. **Lingkungan & Dependensi:** Penyiapan NLTK, Scikit-Learn, PySastrawi, dan Pandas.
2. **Pemuatan Korpus Putusan:** Ingesti 50 berkas teks dari direktori `data/03_txt/`.
3. **Pipeline Preprocessing Bahasa Indonesia:**
   - *Case Folding*: Penyeragaman karakter huruf kecil.
   - *Pembersihan Teks (Regex)*: Eliminasi watermark, nomor lembar, angka, dan tanda baca khusus.
   - *Tokenisasi*: Pemecahan teks panjang menjadi daftar kata individual.
   - *Stopword Filtering*: Pemfilteran korpus stopword Bahasa Indonesia + istilah prosedural hukum peradilan.
   - *Stemming Morfologi*: Pengurangan kata imbuhan ke akar kata dasar (PySastrawi).
4. **Pembobotan Term (TF-IDF):** Perhitungan Term Frequency - Inverse Document Frequency unigram & bigram.
5. **Konstruksi Inverted Index:** Penyusunan struktur berkas terbalik berindeks term -> posting list dokumen berbobot.
6. **Search Engine & Ranking:** Modul pencarian kemiripan kosinus (Cosine Similarity Vector Space Model).
7. **Laporan & Evaluasi Kinerja:** Analisis statistik densitas matriks, kosakata unik, dan ekspor metrik.


---
## 1. Setup Dependensi & Inisialisasi Environment
Memverifikasi modul NLP, mengunduh data leksikon NLTK, dan menghubungkan Google Drive jika dieksekusi via Colab.

In [ ]:
# ======================================================================
# TUGAS 4: Preprocessing, Feature Extraction & Inverted Indexing
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 1: Persiapan Modul dan Struktur Path Sistem
import sys
import subprocess
import platform
from pathlib import Path

ENV_IS_COLAB = 'google.colab' in sys.modules
print(f"[STATUS] Runtime: {'Google Colab' if ENV_IS_COLAB else platform.system()} | Python: {sys.version.split()[0]}")

MODUL_NLP = [
    ('nltk', 'nltk'),
    ('sklearn', 'scikit-learn'),
    ('Sastrawi', 'PySastrawi'),
    ('pandas', 'pandas'),
]

for mod, pkg in MODUL_NLP:
    try:
        __import__(mod)
        print(f"[DEP] {pkg} siap digunakan.")
    except ImportError:
        print(f"[INSTALL] Memasang paket {pkg}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import nltk
for korpus in ['stopwords', 'punkt', 'punkt_tab']:
    try:
        nltk.data.find(f"tokenizers/{korpus}" if 'punkt' in korpus else f"corpora/{korpus}")
    except LookupError:
        nltk.download(korpus, quiet=True)
        print(f"[NLTK] Resource {korpus} berhasil diunduh.")

if ENV_IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    FOLDER_ROOT = Path('/content/drive/MyDrive/PN JEMBER')
else:
    FOLDER_ROOT = Path('.')

LOKASI_DATA = {
    'txt': FOLDER_ROOT / 'data' / '03_txt',
    'hasil': FOLDER_ROOT / 'data' / '04_hasil',
    'index': FOLDER_ROOT / 'data' / '05_index',
}
LOKASI_DATA['index'].mkdir(parents=True, exist_ok=True)

print("[SELESAI] Konfigurasi environment selesai.")
for k, p in LOKASI_DATA.items():
    print(f"- Direktori {k:6s}: {p.resolve()}")


---
## 2. Ingesti Korpus Dokumen Putusan Peradilan
Membaca file teks hasil konversi PDF dari Tugas 3. Seluruh teks dokumen dijadikan bahan pemrosesan sistem TKI.

In [ ]:
# ======================================================================
# TUGAS 4: Preprocessing, Feature Extraction & Inverted Indexing
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 2: Pemuatan Dataset Dokumen Teks Putusan
import sys
from pathlib import Path
import pandas as pd

if 'LOKASI_DATA' not in globals():
    ENV_IS_COLAB = 'google.colab' in sys.modules
    FOLDER_ROOT = Path('/content/drive/MyDrive/PN JEMBER') if ENV_IS_COLAB else Path('.')
    LOKASI_DATA = {
        'txt': FOLDER_ROOT / 'data' / '03_txt',
        'hasil': FOLDER_ROOT / 'data' / '04_hasil',
        'index': FOLDER_ROOT / 'data' / '05_index',
    }
    LOKASI_DATA['index'].mkdir(parents=True, exist_ok=True)

koleksi_dokumen = []

if LOKASI_DATA['txt'].exists():
    daftar_berkas_txt = sorted(LOKASI_DATA['txt'].glob('*.txt'))
    print(f"[DATA] Mendeteksi {len(daftar_berkas_txt)} berkas teks putusan di {LOKASI_DATA['txt']}")
    for no_urut, file_p in enumerate(daftar_berkas_txt, 1):
        try:
            isi_teks = file_p.read_text(encoding='utf-8', errors='replace')
        except Exception:
            isi_teks = file_p.read_text(encoding='latin-1', errors='replace')
        koleksi_dokumen.append({
            'doc_id': no_urut,
            'nama_file': file_p.name,
            'teks_mentah': isi_teks,
        })

if not koleksi_dokumen:
    print("[FALLBACK] Folder TXT belum ada. Mengambil dari data CSV hasil ekstraksi...")
    file_csv = LOKASI_DATA['hasil'] / 'hasil_ekstraksi.csv'
    if file_csv.exists():
        df_c = pd.read_csv(file_csv).fillna('')
        for idx, r in df_c.iterrows():
            isi_gabung = f"{r.get('barang_bukti', '')} {r.get('amar_putusan', '')}"
            koleksi_dokumen.append({
                'doc_id': idx + 1,
                'nama_file': str(r.get('file', f'dokumen_{idx+1}.txt')),
                'teks_mentah': isi_gabung,
            })

df_dataset = pd.DataFrame(koleksi_dokumen)
print(f"[TOTAL] {len(df_dataset)} dokumen berhasil dimuat ke memori kerja.")

if not df_dataset.empty:
    sebaran_pjg = df_dataset['teks_mentah'].str.len()
    print(f"- Rerata Karakter per Dokumen : {sebaran_pjg.mean():.0f}")
    print(f"- Dokumen Terpanjang          : {sebaran_pjg.max():,} karakter ({df_dataset.loc[sebaran_pjg.idxmax(), 'nama_file']})")
    print(f"- Dokumen Terpendek           : {sebaran_pjg.min():,} karakter ({df_dataset.loc[sebaran_pjg.idxmin(), 'nama_file']})")
    print("-" * 70)
    print("Pratinjau Awal Dokumen Pertama (400 karakter):")
    print(df_dataset.iloc[0]['teks_mentah'][:400].strip())
    print("...")


---
## 3. Pipeline Preprocessing Teks Bahasa Indonesia
Menerapkan tahapan pembersihan berlapis mencakup case folding, pembersihan noise regulasi, tokenisasi, eliminasi stopword kustom hukum, serta reduksi akar kata (stemming).

In [ ]:
# ======================================================================
# TUGAS 4: Preprocessing, Feature Extraction & Inverted Indexing
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 3: Pembersihan, Tokenisasi, Filtering Stopword & Stemming
import sys
import re
import time
from pathlib import Path
import pandas as pd

if 'df_dataset' not in globals():
    raise RuntimeError("Harap jalankan Sel 2 terlebih dahulu.")

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Inisialisasi Stemmer PySastrawi dengan fallback terarah
try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    factory = StemmerFactory()
    stemmer_sastrawi = factory.create_stemmer()
    def lematisasi_akar(kata):
        return stemmer_sastrawi.stem(kata)
    print("[STEMMER] PySastrawi Stemmer aktif secara penuh.")
except ImportError:
    SUFIKS_ID = ['kan', 'an', 'i', 'nya', 'lah', 'kah', 'pun', 'ku', 'mu']
    def lematisasi_akar(kata):
        for s in SUFIKS_ID:
            if kata.endswith(s) and len(kata) - len(s) >= 3:
                return kata[:-len(s)]
        return kata
    print("[STEMMER] Memakai fallback suffix-stripping sederhana.")

# Korpus Kata Henti Bahasa Indonesia + Istilah Prosedural Perkara
STOPWORDS_DASAR = set(stopwords.words('indonesian'))
STOPWORDS_HUKUM = {
    'putusan', 'pengadilan', 'negeri', 'nomor', 'tahun', 'halaman', 'bahwa',
    'dalam', 'dengan', 'atas', 'oleh', 'kepada', 'dari', 'untuk', 'pada', 'yang',
    'telah', 'akan', 'adalah', 'ini', 'itu', 'dan', 'atau', 'juga', 'serta', 'lebih',
    'maka', 'sebagai', 'tidak', 'sesuai', 'berdasarkan', 'undang', 'pasal', 'ayat',
    'huruf', 'angka', 'ketentuan', 'tentang', 'jo', 'junto', 'pid', 'sus', 'pn',
    'jmr', 'jember', 'mahkamah', 'agung', 'republik', 'indonesia', 'demi',
    'keadilan', 'ketuhanan', 'maha', 'esa', 'hakim', 'ketua', 'majelis',
    'panitera', 'pengganti', 'jaksa', 'penuntut', 'umum', 'terdakwa', 'saksi',
    'keterangan', 'surat', 'dakwaan', 'tuntutan', 'pembelaan', 'replik', 'duplik',
    'mengadili', 'menyatakan', 'menghukum', 'membebaskan', 'menetapkan',
    'diucapkan', 'sidang', 'terbuka', 'hari', 'tanggal', 'bulan',
    'memperhatikan', 'menimbang', 'mengingat', 'memutuskan', 'pertimbangan',
    'hukum', 'fakta', 'yuridis', 'sosiologis', 'keadaan', 'memberatkan',
    'meringankan', 'terdaftar', 'register', 'perkara', 'jenis', 'kelamin',
    'laki', 'perempuan', 'agama', 'islam', 'kewarganegaraan', 'pekerjaan',
    'tempat', 'tinggal', 'umur', 'ditahan', 'ditangguhkan', 'biaya', 'dibebankan'
}
SEMUA_STOPWORDS = STOPWORDS_DASAR | STOPWORDS_HUKUM

def transformasi_case_folding(t):
    return t.lower()

def eliminasi_noise_teks(t):
    t = re.sub(r'(?i)disclaimer\s*kepaniteraan.*', ' ', t)
    t = re.sub(r'(?i)halaman\s+\d+\s+dari\s+\d+', ' ', t)
    t = re.sub(r'(?i)mahkamah agung republik indonesia', ' ', t)
    t = re.sub(r'[^a-zA-Z\s]', ' ', t)           # Hanya pertahankan karakter alfabet
    t = re.sub(r'\b\w{1,2}\b', ' ', t)             # Buang token 1-2 karakter tidak bermakna
    t = re.sub(r'\s{2,}', ' ', t)                 # Normalisasi spasi ganda
    return t.strip()

def pemecah_token(t):
    try:
        return word_tokenize(t, language='indonesian')
    except Exception:
        return t.split()

def tapis_kata_henti(tokens):
    return [w for w in tokens if w not in SEMUA_STOPWORDS and len(w) >= 3]

def reduksi_kata_akar(tokens):
    return [lematisasi_akar(w) for w in tokens]

def jalankan_pipeline_preprocessing(teks_mentah):
    t = transformasi_case_folding(teks_mentah)
    t = eliminasi_noise_teks(t)
    tok = pemecah_token(t)
    tok = tapis_kata_henti(tok)
    tok = reduksi_kata_akar(tok)
    return tok

print(f"[PROSES] Menjalankan preprocessing terhadap {len(df_dataset)} dokumen...")
t_awal = time.time()
hasil_tokenisasi = []
for _, row in df_dataset.iterrows():
    tokens_dok = jalankan_pipeline_preprocessing(row['teks_mentah'])
    hasil_tokenisasi.append(tokens_dok)

df_dataset['tokens'] = hasil_tokenisasi
t_durasi = time.time() - t_awal
print(f"[SELESAI] Preprocessing tuntas dalam {t_durasi:.2f} detik.")

pjg_token = df_dataset['tokens'].apply(len)
print(f"- Rata-rata Token Bersih : {pjg_token.mean():.0f} token/dokumen")
print(f"- Dokumen Token Terbanyak: {pjg_token.max()} token ({df_dataset.loc[pjg_token.idxmax(), 'nama_file']})")
print(f"- Dokumen Token Tersedikit: {pjg_token.min()} token ({df_dataset.loc[pjg_token.idxmin(), 'nama_file']})")
print("-" * 70)
print("Contoh 20 Token Hasil Pemrosesan Dokumen Pertama:")
print(df_dataset.iloc[0]['tokens'][:20])


---
## 4. Ekstraksi Fitur & Pembobotan Term (TF-IDF Matrix)
Membangun representasi vektor dari seluruh korpus dengan model pembobotan TF-IDF berskala sublinear.

In [ ]:
# ======================================================================
# TUGAS 4: Preprocessing, Feature Extraction & Inverted Indexing
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 4: Pembobotan TF-IDF Unigram & Bigram
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

if 'df_dataset' not in globals() or 'tokens' not in df_dataset.columns:
    raise RuntimeError("Jalankan Sel 3 terlebih dahulu.")

# Rekonstruksi kalimat hasil preprocessing untuk input TF-IDF
korpus_bersih = df_dataset['tokens'].apply(lambda arr: ' '.join(arr)).tolist()

vektorisator_tfidf = TfidfVectorizer(
    min_df=1,
    max_df=0.92,
    sublinear_tf=True,
    ngram_range=(1, 2),
)

matriks_tfidf = vektorisator_tfidf.fit_transform(korpus_bersih)
fitur_kosakata = vektorisator_tfidf.get_feature_names_out()

print("=" * 70)
print("HASIL PEMBOBOTAN TF-IDF MATRIKS")
print("=" * 70)
print(f"Dimensi Matriks      : {matriks_tfidf.shape[0]} Dokumen x {matriks_tfidf.shape[1]} Term Unik")
print(f"Total Nilai Non-Nol  : {matriks_tfidf.nnz:,}")
densitas = (matriks_tfidf.nnz / (matriks_tfidf.shape[0] * matriks_tfidf.shape[1])) * 100
print(f"Densitas Matriks     : {densitas:.3f}%")

arr_matriks = matriks_tfidf.toarray()
print("-" * 70)
print("Daftar 8 Term dengan Bobot TF-IDF Tertinggi pada Dokumen 1 s.d. 3:")
for doc_idx in range(min(3, len(df_dataset))):
    bobot_doc = arr_matriks[doc_idx]
    top_indices = np.argsort(bobot_doc)[::-1][:8]
    print(f"\n[Dokumen {doc_idx+1}] {df_dataset.iloc[doc_idx]['nama_file']}:")
    for rank, idx_term in enumerate(top_indices, 1):
        if bobot_doc[idx_term] > 0:
            print(f"   {rank}. {fitur_kosakata[idx_term]:25s} | Bobot: {bobot_doc[idx_term]:.4f}")


---
## 5. Konstruksi Berkas Indeks Terbalik (Inverted Index)
Membangun struktur data pengindeksan term-ke-dokumen (Posting List) beserta frekuensi term dan bobot TF-IDF, lalu menyimpannya dalam format JSON.

In [ ]:
# ======================================================================
# TUGAS 4: Preprocessing, Feature Extraction & Inverted Indexing
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 5: Pembangunan & Serialisasi Inverted Index
import json
from collections import defaultdict
from pathlib import Path

if 'matriks_tfidf' not in globals():
    raise RuntimeError("Jalankan Sel 4 terlebih dahulu.")

berkas_indeks_terbalik = defaultdict(list)

print("[INDEKS] Membangun posting list Inverted Index...")
for row_i, r in df_dataset.iterrows():
    id_dok = int(r['doc_id'])
    nama_f = r['nama_file']
    daftar_t = r['tokens']

    frek_term = defaultdict(int)
    for t in daftar_t:
        frek_term[t] += 1

    bobot_vektor = arr_matriks[row_i]
    for term, count in frek_term.items():
        idx_vocab = vektorisator_tfidf.vocabulary_.get(term, -1)
        skor_tfidf = round(float(bobot_vektor[idx_vocab]), 6) if idx_vocab >= 0 else 0.0

        if count > 0 or skor_tfidf > 0:
            berkas_indeks_terbalik[term].append({
                'doc_id': id_dok,
                'nama_file': nama_f,
                'tf': count,
                'tfidf': skor_tfidf,
            })

# Urutkan setiap posting list berdasarkan bobot TF-IDF terbesar
for k in berkas_indeks_terbalik:
    berkas_indeks_terbalik[k].sort(key=lambda item: item['tfidf'], reverse=True)

berkas_indeks_dict = dict(berkas_indeks_terbalik)

FILE_JSON_INDEX = LOKASI_DATA['index'] / 'inverted_index.json'
with open(FILE_JSON_INDEX, 'w', encoding='utf-8') as fj:
    json.dump(berkas_indeks_dict, fj, ensure_ascii=False, indent=2)

ukuran_file_kb = FILE_JSON_INDEX.stat().st_size / 1024
print(f"[SELESAI] Inverted Index berhasil dibangun!")
print(f"- Total Entri Term Unik : {len(berkas_indeks_dict):,} kata")
print(f"- Lokasi Berkas JSON    : {FILE_JSON_INDEX.resolve()} ({ukuran_file_kb:.1f} KB)")

print("-" * 70)
print("Sampel Posting List untuk Kata Kunci Substantif:")
for contoh_term in ['sabu', 'ganja', 'timbangan', 'ekstasi', 'kurir']:
    post = berkas_indeks_dict.get(contoh_term, [])
    if post:
        print(f"\n* Term '{contoh_term}' ditemukan pada {len(post)} dokumen. Top 2 posting:")
        for p in post[:2]:
            print(f"    -> Doc {p['doc_id']:02d} ({p['nama_file']}): TF={p['tf']} | TF-IDF={p['tfidf']:.4f}")
    else:
        print(f"\n* Term '{contoh_term}' tidak muncul dalam korpus.")


---
## 6. Mesin Pencari Dokumen (Vector Space Model & Cosine Similarity)
Mengimplementasikan fungsi pencarian kuari pengguna yang diproses melalui pipeline preprocessing yang sama dan diranking berdasarkan derajat kedekatan vektor dokumen.

In [ ]:
# ======================================================================
# TUGAS 4: Preprocessing, Feature Extraction & Inverted Indexing
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 6: Algoritma Pencarian & Pemeringkatan Putusan
import re
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

if 'vektorisator_tfidf' not in globals() or 'matriks_tfidf' not in globals():
    raise RuntimeError("Jalankan Sel 4 terlebih dahulu.")

def praproses_kuari(kuari_teks):
    q = kuari_teks.lower()
    q = re.sub(r'[^a-zA-Z\s]', ' ', q)
    q = re.sub(r'\s{2,}', ' ', q).strip()
    kata_list = [lematisasi_akar(w) for w in q.split() if w not in SEMUA_STOPWORDS and len(w) >= 2]
    return ' '.join(kata_list)

def mesin_pencari_putusan(kuari, top_k=5, cetak_hasil=True):
    kuari_terproses = praproses_kuari(kuari)
    if not kuari_terproses.strip():
        if cetak_hasil:
            print(f"[PERINGATAN] Kuari '{kuari}' tidak menghasilkan token valid.")
        return pd.DataFrame()

    # Transformasi vektor kuari ke ruang fitur yang sama
    vektor_kuari = vektorisator_tfidf.transform([kuari_terproses])
    derajat_mirip = cosine_similarity(vektor_kuari, matriks_tfidf).flatten()

    indeks_terpilih = np.argsort(derajat_mirip)[::-1][:top_k]

    baris_output = []
    for urut, idx in enumerate(indeks_terpilih, 1):
        skor = float(derajat_mirip[idx])
        if skor <= 0.0:
            break
        item_dok = df_dataset.iloc[idx]
        teks_penuh = item_dok['teks_mentah']

        cuplikan = '-'
        for k_term in kuari.lower().split():
            pos = teks_penuh.lower().find(k_term)
            if pos >= 0:
                mulai = max(0, pos - 60)
                selesai = min(len(teks_penuh), pos + 100)
                cuplikan = f"...{teks_penuh[mulai:selesai].strip()}..."
                break

        baris_output.append({
            'Peringkat': urut,
            'Doc_ID': int(item_dok['doc_id']),
            'Nama_Berkas': item_dok['nama_file'],
            'Skor_Kosinus': round(skor, 4),
            'Cuplikan_Teks': cuplikan,
        })

    df_hasil_cari = pd.DataFrame(baris_output)
    if cetak_hasil:
        print(f"\n[PENCARIAN] Kuari Asli: '{kuari}' | Kuari Normalisasi: '{kuari_terproses}'")
        if df_hasil_cari.empty:
            print("-> Tidak ditemukan dokumen yang relevan.")
        else:
            pd.set_option('display.max_colwidth', 65)
            print(df_hasil_cari[['Peringkat', 'Doc_ID', 'Nama_Berkas', 'Skor_Kosinus']].to_string(index=False))
    return df_hasil_cari

print("=" * 70)
print("PENGUJIAN MESIN TEMU KEMBALI INFORMASI (TUGAS 4)")
print("=" * 70)

KUMPULAN_KUARI_UJI = [
    "sabu sabu gram narkotika",
    "ganja kering bungkus kertas",
    "timbangan digital plastik klip",
    "pemufakatan jahat tanpa hak",
    "rehabilitasi medis pecandu narkoba",
]

for q_uji in KUMPULAN_KUARI_UJI:
    mesin_pencari_putusan(q_uji, top_k=3)


---
## 7. Evaluasi & Ekspor Laporan Statistik Sistem
Menyajikan ringkasan metrik korpus hasil ekstraksi dan menyimpan rekapitulasi ke file CSV `statistik_index.csv`.

In [ ]:
# ======================================================================
# TUGAS 4: Preprocessing, Feature Extraction & Inverted Indexing
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 7: Rekapitulasi Metrik Evaluasi Sistem Pengindeksan
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd

if 'df_dataset' not in globals() or 'berkas_indeks_dict' not in globals():
    raise RuntimeError("Jalankan seluruh sel sebelumnya terlebih dahulu.")

kumpulan_seluruh_token = [tok for sublist in df_dataset['tokens'] for tok in sublist]
frek_seluruh_kata = Counter(kumpulan_seluruh_token)
panjang_posting_list = [len(v) for v in berkas_indeks_dict.values()]

print("=" * 70)
print("RINGKASAN AKHIR EVALUASI TKI - TUGAS 4")
print("=" * 70)
print(f"1. Kuantitas Dokumen Putusan      : {len(df_dataset)} berkas")
print(f"2. Total Token Terproses          : {len(kumpulan_seluruh_token):,} token")
print(f"3. Rerata Token per Dokumen       : {np.mean(df_dataset['tokens'].apply(len)):.0f} token")
print(f"4. Jumlah Entri Inverted Index    : {len(berkas_indeks_dict):,} term unik")
print(f"5. Dimensi Matriks TF-IDF         : {matriks_tfidf.shape[0]} x {matriks_tfidf.shape[1]}")
print(f"6. Rerata Panjang Posting per Term: {np.mean(panjang_posting_list):.2f} dokumen")

print("-" * 70)
print("10 Term Substantif Paling Dominan:")
top_10_kata = pd.DataFrame(frek_seluruh_kata.most_common(10), columns=['Term', 'Frekuensi_Total'])
top_10_kata.index = range(1, len(top_10_kata) + 1)
print(top_10_kata.to_string())

# Ekspor metrik evaluasi ke berkas CSV
METRIK_OUTPUT = {
    'total_dokumen': len(df_dataset),
    'total_token': len(kumpulan_seluruh_token),
    'rata_token_per_dok': round(float(np.mean(df_dataset['tokens'].apply(len)))),
    'total_term_indeks': len(berkas_indeks_dict),
    'tfidf_rows': matriks_tfidf.shape[0],
    'tfidf_cols': matriks_tfidf.shape[1],
    'tfidf_nonzero': matriks_tfidf.nnz,
    'rata_posting_length': round(float(np.mean(panjang_posting_list)), 3),
}

FILE_STAT_CSV = LOKASI_DATA['index'] / 'statistik_index.csv'
pd.DataFrame([METRIK_OUTPUT]).to_csv(FILE_STAT_CSV, index=False, encoding='utf-8-sig')
print(f"\n[EKSPOR] Laporan statistik tersimpan di: {FILE_STAT_CSV.resolve()}")
print("=" * 70)
print("[SELESAI] Tugas 4 Sukses Dijalankan Tanpa Hambatan.")
